# score_runs_baseline_val — score the step-0 anchor on val
Canonical scorer on the base-model val images. Writes `val__baseline_val.jsonl` and prints the mean PF/SC/Overall — the measured step-0 point the early SC curve is compared against. Rerun until `to score 0`.

## 1. Repo, scorer patch

In [ ]:
import os, subprocess, sys
REPO = "/content/OmniGen2"
if not os.path.isdir(REPO):
    subprocess.run(["git","clone","https://github.com/VectorSpaceLab/OmniGen2.git",REPO], check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","regex"], check=True)   # json_util imports `regex`
sys.path.insert(0, REPO)
os.chdir(REPO)

# without the break, evaluate() re-runs a successful parse to max_tries: 6 calls per row instead of 2
p = "/content/OmniGen2/omnicontext/omnicontext_score.py"
lines = open(p).read().split("\n")
if not any("added: stop after successful parse" in l for l in lines):
    out = []
    for ln in lines:
        out.append(ln)
        if 'raise Exception("rate_limit_exceeded")' in ln:
            out.append("                break   # added: stop after successful parse")
    open(p, "w").write("\n".join(out))
    print("patched")
else:
    print("already patched")

patched


## 2. Drive, run config

In [ ]:
from google.colab import drive, userdata
import os, json
drive.mount("/content/drive")
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

DRIVE    = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
DATASET  = "val"
RUN      = "baseline_val"     # measured step-0 anchor
gen = f"{DRIVE}/gen/{DATASET}/{RUN}"
n = len([f for f in os.listdir(gen) if f.endswith(".png")]) if os.path.isdir(gen) else 0
print(f"{RUN}: {n} images at {gen}")

Mounted at /content/drive
baseline_val: 98 images at /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/gen/val/baseline_val


## 3. Reference images

In [ ]:
IMGS, EXPECTED = "/content/imgs", 37063
TAR = f"{DRIVE}/imgs_subset.tar"

def n_files(d): return sum(len(f) for _, _, f in os.walk(d)) if os.path.isdir(d) else 0

if n_files(IMGS) < EXPECTED:
    os.makedirs(IMGS, exist_ok=True)
    r = subprocess.run(f'cp "{TAR}" /content/imgs_subset.tar && tar -xf /content/imgs_subset.tar -C {IMGS}',
                       shell=True, capture_output=True, text=True)
    if r.returncode: raise RuntimeError(f"rc={r.returncode}\n{r.stderr[:800]}")
    print("restored", n_files(IMGS), "/", EXPECTED)
    assert n_files(IMGS) >= EXPECTED, "restore incomplete"
else:
    print("images complete:", n_files(IMGS))

for s in ["train", "val", "test", "screen", "screen_small"]:
    src = f"{DRIVE}/{s}.jsonl"
    if os.path.exists(src) and not os.path.exists(f"/content/{s}.jsonl"):
        subprocess.run(["cp", src, f"/content/{s}.jsonl"])

rows = [json.loads(l) for l in open(f"/content/{DATASET}.jsonl", encoding="utf-8")]
# indices generated across all step dirs (each step scored separately in the loop below)
all_have = sorted(int(f[:-4]) for f in os.listdir(f"{DRIVE}/gen/{DATASET}/{RUN}") if f.endswith(".png") and f[:-4].isdigit())
assert all_have, f"no generated images found for {RUN}"
assert max(all_have) < len(rows), f"image index {max(all_have)} outside {DATASET} ({len(rows)} rows)"
missing = [p for i in all_have for p in rows[i]["input_images"] if not os.path.exists(os.path.join(IMGS, p))]
assert not missing, f"{len(missing)} reference images missing, first: {missing[:3]}"
print(f"{DATASET}: {len(rows)} rows | {len(all_have)} generated images")

restored 37063 / 37063
val: 98 rows | 98 generated images


## 4. Sanity — 3 rows

In [ ]:
import math
from PIL import Image
from omnicontext.omnicontext_score import OmniContextScore

scorer = OmniContextScore("https://api.openai.com/v1/chat/completions", userdata.get("OPENAI_API_KEY"))

def resize_ref(im):   # OmniContext default: cap at 1024^2, round to /16 — references only
    w, h = im.size
    cap = 1024*1024
    if w*h > cap:
        r = math.sqrt(cap/(w*h)); w, h = int(w*r), int(h*r)
    return im.resize((max(16, w//16*16), max(16, h//16*16)))

# 3-row smoke on the first step's images to confirm the scorer + adapter path before the full loop
_smoke_GEN  = f"{DRIVE}/gen/{DATASET}/{RUN}"
_smoke_have = sorted(int(f[:-4]) for f in os.listdir(_smoke_GEN) if f.endswith(".png") and f[:-4].isdigit())
for i in _smoke_have[:3]:
    r = rows[i]
    refs = [resize_ref(Image.open(os.path.join(IMGS, p)).convert("RGB")) for p in r["input_images"]]
    gen  = Image.open(f"{_smoke_GEN}/{i:05d}.png").convert("RGB")
    res  = scorer.evaluate(refs + [gen], r["instruction"], with_scene=False)
    print(f"row {i}: PF={res['PF_scores']['score']}  SC={res['SC_scores']['score']}")
    print(f"   PF: {str(res['PF_scores']['reasoning'])[:140]}")
    print(f"   SC: {str(res['SC_scores']['reasoning'])[:140]}\n")

row 0: PF=9  SC=8
   PF: The image accurately depicts a professional man standing confidently next to a sleek, open laptop that is displaying a vibrant presentation.
   SC: The man in the final image (image 2) is clearly intended to be the same as the individual in the reference image (image 1). The facial struc

row 1: PF=9  SC=10
   PF: The image effectively fulfills most of the detailed instructions: a professional man in a tailored suit is standing confidently next to a la
   SC: The laptop in the final image perfectly matches the laptop from the reference image in terms of color, keyboard layout, and screen. No incon

row 2: PF=8  SC=8
   PF: The final image effectively depicts a professional man standing confidently next to (actually holding) a sleek laptop. He is dressed in a ta
   SC: The man in the final image closely matches the subject in image 1 in terms of facial features, hairstyle, and overall appearance, and is wea



## 5. Full pass (resumable)

In [ ]:
import time, math
from PIL import Image
from concurrent.futures import ThreadPoolExecutor, as_completed

def num(x):
    if isinstance(x, list) and x: x = x[0]
    return x if isinstance(x, (int, float)) else None

GEN    = f"{DRIVE}/gen/{DATASET}/{RUN}"
SCORES = f"{DRIVE}/scores/{DATASET}__{RUN}.jsonl"
have   = sorted(int(f[:-4]) for f in os.listdir(GEN) if f.endswith(".png") and f[:-4].isdigit())
done = set()
if os.path.exists(SCORES):
    for l in open(SCORES, encoding="utf-8"):
        try:
            d = json.loads(l)
            if "error" not in d: done.add(d["i"])
        except: pass

def score_row(i):
    r  = rows[i]
    gp = f"{GEN}/{i:05d}.png"
    if not os.path.exists(gp): return {"i": i, "error": "missing_generated"}
    refs = [resize_ref(Image.open(os.path.join(IMGS, p)).convert("RGB")) for p in r["input_images"]]
    gen  = Image.open(gp).convert("RGB")
    time.sleep(0.2)
    res = scorer.evaluate(refs + [gen], r["instruction"], with_scene=False)
    pf, sc = num(res["PF_scores"]["score"]), num(res["SC_scores"]["score"])
    if pf is None or sc is None: return {"i": i, "error": "unparsable_score"}
    if any(str(res[k]["reasoning"]).startswith("guess_if_cannot_parse") for k in ("PF_scores", "SC_scores")):
        return {"i": i, "error": "guessed_score"}
    ups = ["_up" in os.path.basename(p) for p in r["input_images"]]
    return {"i": i, "uid": r["output_image"] + "|" + "|".join(sorted(r["input_images"])), "n_refs": len(r["input_images"]),
            "ref_kind": "all_up" if all(ups) else ("no_up" if not any(ups) else "mixed"),
            "PF": pf, "SC": sc,
            "PF_reason": res["PF_scores"]["reasoning"], "SC_reason": res["SC_scores"]["reasoning"]}

todo = [i for i in have if i not in done]
print(f"to score {len(todo)} (done {len(done)}) -> {SCORES}")
with open(SCORES, "a", encoding="utf-8") as f, ThreadPoolExecutor(max_workers=8) as ex:
    futs = [ex.submit(score_row, i) for i in todo]
    for n, fut in enumerate(as_completed(futs), 1):
        f.write(json.dumps(fut.result(), ensure_ascii=False) + "\n"); f.flush()
        if n % 25 == 0: print(f"  {n}/{len(todo)}")
print("done — rerun until 'to score 0'")

to score 98 (done 0) -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/scores/val__baseline_val.jsonl
  25/98
  50/98
  75/98
done — rerun until 'to score 0'


## 6. Aggregate — step-0 anchor

In [ ]:
import math, json
import numpy as np
ov = lambda r: math.sqrt(max(r["PF"],0)*max(r["SC"],0))
rs = [json.loads(l) for l in open(f"{DRIVE}/scores/{DATASET}__{RUN}.jsonl", encoding="utf-8") if "error" not in json.loads(l)]
print(f"baseline-on-val (step-0 anchor): {len(rs)} rows")
print(f"  PF {np.mean([r['PF'] for r in rs]):.3f}  SC {np.mean([r['SC'] for r in rs]):.3f}  Overall {np.mean([ov(r) for r in rs]):.3f}")
print("\nthis is the measured step-0 point; the early curve (250/500/750/1000) is scored above.")

baseline-on-val (step-0 anchor): 98 rows
  PF 7.898  SC 7.816  Overall 7.739

this is the measured step-0 point; the early curve (250/500/750/1000) is scored above.
